In [ ]:
import subprocess
import os
import time
import re
import threading
import datetime
import sys
import glob
import json
from IPython.display import display, HTML

SSH_PORT = 2222
ROOT_PASSWORD = "Zavin123"
AFISLA_LOG = "afisla.log"

current_relay_port = None
sshd_process = None
afisla_process = None

def cleanup_old_sessions():
    """Membersihkan sesi dan file log lama."""
    print("\n[CLEANUP] Menghentikan semua sesi lama...")
    services = ["cloudflared", "afisla", "lt", "serveo", "sshd"]
    for s in services:
        subprocess.run(["pkill", "-f", s], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    logs = ["cf.log", "afisla.log", "sshd_config_custom"]
    for l in logs:
        if os.path.exists(l):
            try:
                os.remove(l)
            except Exception:
                pass
    time.sleep(3)
    print("[OK] Sesi lama dibersihkan.")

def check_and_install_dependencies():
    """Memeriksa dan menginstal tools/dependensi yang dibutuhkan."""
    print("[+] Memeriksa dependensi sistem...")

    packages = ["openssh-server", "curl"]
    needed_pkgs = []

    for pkg in packages:
        res = subprocess.run(["dpkg", "-s", pkg], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        if res.returncode != 0:
            needed_pkgs.append(pkg)

    if needed_pkgs:
        print(f"[+] Menginstall: {', '.join(needed_pkgs)}")
        subprocess.run(["apt-get", "update", "-qq"], check=True)
        subprocess.run(["apt-get", "install", "-y", "-qq"] + needed_pkgs, check=True)

    if subprocess.run(["which", "afisla"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode != 0:
        print("[+] Menginstall Afisla Tunnel Client...")
        os.system("curl -fsSL https://afisla.web.id/install.sh | bash")

    print("[OK] Semua dependensi siap.")

def set_root_password():
    """Set password untuk root."""
    print("[+] Setting root password...")
    p = subprocess.Popen(["echo", f"root:{ROOT_PASSWORD}"], stdout=subprocess.PIPE)
    subprocess.run(["chpasswd"], stdin=p.stdout, check=True)
    p.stdout.close()
    print(f"[OK] Password root: {ROOT_PASSWORD}")

def setup_sshd():
    """Membuat konfig sshd_custom dan menjalankan daemon sshd."""
    global sshd_process
    print("[+] Mengonfigurasi SSH Server...")

    ssh_config_content = f"""Port {SSH_PORT}
PermitRootLogin yes
PasswordAuthentication yes
PubkeyAuthentication no
ChallengeResponseAuthentication no
UsePAM yes
LogLevel VERBOSE
ClientAliveInterval 30
ClientAliveCountMax 3
TCPKeepAlive yes
Subsystem sftp /usr/lib/openssh/sftp-server
"""
    config_file = os.path.abspath("sshd_config_custom")
    with open(config_file, "w") as f:
        f.write(ssh_config_content)

    os.makedirs("/var/run/sshd", exist_ok=True)
    os.makedirs("/run/sshd", exist_ok=True)
    subprocess.run(["chmod", "0755", "/var/run/sshd"], check=False)

    for key_file in glob.glob("/etc/ssh/ssh_host_*"):
        try:
            os.remove(key_file)
        except Exception:
            pass

    subprocess.run(["ssh-keygen", "-A"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=False)

    sshd_process = subprocess.Popen(['/usr/sbin/sshd', '-D', '-f', config_file],
                                    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(2)
    print(f"[OK] SSH Server berjalan di port {SSH_PORT}.")

def start_afisla():
    """Jalankan Afisla Tunnel Client."""
    global afisla_process
    print("[+] Menjalankan Afisla Tunnel Client...")
    afisla_log_file = open(AFISLA_LOG, "w")
    afisla_process = subprocess.Popen(['afisla', 'client', '--port-local', str(SSH_PORT)],
                                      stdout=afisla_log_file, stderr=subprocess.STDOUT)

def monitor_services():
    """Thread monitoring untuk auto-restart dan deteksi port relay."""
    global sshd_process, afisla_process, current_relay_port
    config_file = os.path.abspath("sshd_config_custom")

    while True:
        time.sleep(10)

        sys.stdout.write('\x00')
        sys.stdout.flush()

        if sshd_process and sshd_process.poll() is not None:
            sshd_process = subprocess.Popen(['/usr/sbin/sshd', '-D', '-f', config_file],
                                            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

        if os.path.exists(AFISLA_LOG):
            with open(AFISLA_LOG, "r") as f:
                log = f.read()
                match = re.search(r"(?:port|relay)[:\s]+(\d{4,5})", log, re.IGNORECASE) or re.search(r"(\d{5})", log)
                if match:
                    new_port = match.group(1)
                    if new_port != current_relay_port:
                        current_relay_port = new_port
                        ssh_cmd = f"ssh -o StrictHostKeyChecking=no -o UserKnownHostsFile=/dev/null -o ServerAliveInterval=30 -o PreferredAuthentications=password -o ProxyCommand='nc relay.afisla.web.id {current_relay_port}' root@127.0.0.1 -p {SSH_PORT}"
                        ssh_cmd_js = json.dumps(ssh_cmd)
                        uid = str(int(time.time()*1000))
                        print("\n" + "="*70)
                        print("[AFISLA TUNNEL READY]")
                        print(f"User      : root")
                        print(f"Password  : {ROOT_PASSWORD}")
                        print(f"Port Relay: {current_relay_port}")
                        print("="*70)
                        display(HTML(f"""
                        <div style="background:#1e1e2e;border-radius:10px;padding:16px;margin:10px 0;border:1px solid #313244;">
                          <div style="color:#f38ba8;font-weight:bold;font-size:14px;margin-bottom:8px;">SSH Command</div>
                          <textarea id="cmd_{uid}" readonly style="width:100%;background:#181825;color:#a6e3a1;border:1px solid #45475a;border-radius:6px;padding:10px;font-family:monospace;font-size:12px;resize:none;outline:none;cursor:pointer;" rows="2" onclick="this.select()">{ssh_cmd}</textarea>
                          <button onclick="var t=document.getElementById('cmd_{uid}');t.select();document.execCommand('copy');this.innerText='Copied!';var b=this;setTimeout(function(){{b.innerText='Copy'}},1500)" style="margin-top:8px;background:#f38ba8;color:#1e1e2e;border:none;padding:6px 16px;border-radius:6px;cursor:pointer;font-weight:bold;font-size:13px;">Copy</button>
                        </div>
                        """))

if __name__ == "__main__":
    cleanup_old_sessions()
    check_and_install_dependencies()
    set_root_password()
    setup_sshd()
    start_afisla()

    threading.Thread(target=monitor_services, daemon=True).start()

    print("[+] All services running. Waiting for Afisla tunnel...")

    while True:
        time.sleep(40)
        print(f"[{datetime.datetime.now().strftime('%H:%M:%S')}] Heartbeat: Services running...")
